In [54]:
import pickle
import pandas as pd
from perturbqa.data import load_de

def convert_results_to_csv(
    pred_pkl_path: str,
    dataset: str = "k562",
    threshold: float = 0.5,
    output_csv_path: str = "converted_results.csv"
):
    # Load predictions and ground truth
    with open(pred_pkl_path, "rb") as f:
        pred = pickle.load(f)

    true = load_de(dataset)["test"]

    if 'key' not in pred.keys():
        pred['key'] = pred['de_key']

    keys, predictions, pred_labels = pred['key'], pred['de_pred'], pred['de_true']

    assert len(true) == len(keys) == len(predictions) == len(pred_labels)


    rows = []
    for label, key, output, pred_label in zip(true, keys, predictions, pred_labels):
        assert key[0] == label['pert']
        assert key[1] == label["gene"]
        assert pred_label == label["label"]

        row = {
            "prompt": "",
            "completion": "",
            "answer": output,
            "binary_answer": int(output >= threshold),
            "ground_truth": label["label"],
            "gene_perturbed": label['pert'],
            "gene_monitored": label["gene"],
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(output_csv_path, index=False)
    print(f"✅ Saved {len(df)} rows to {output_csv_path}")

In [55]:
for method in ['gears', 'summer']:
    for cell_line in ['rpe1', 'k562', 'hepg2', 'jurkat']:
        convert_results_to_csv(
            f'/opt/jupyter-envs/rbio/rbio-fmilletari-2/work/PerturbQA/results/{method}/{cell_line}.pkl',
            cell_line,
            0.5,
            f'/mnt/czi-sci-ai/project-rbio/baselines/{method}-{cell_line}.csv'
        )

✅ Saved 23212 rows to /mnt/czi-sci-ai/project-rbio/baselines/summer-k562.csv
